In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

In [ ]:
face = pd.read_csv('data/labeled_WM_Face_2bk_cleaned_analysis_results.csv')
place = pd.read_csv('data/labeled_WM_Place_2bk_cleaned_analysis_results.csv')
SA_rank = pd.read_csv('data/SA_rank.csv')

In [ ]:
region_to_rank = SA_rank.set_index('region')['final.rank']

In [ ]:
dataframes = [face, place]
for df in dataframes:
    # Extract the region name from the 'label' column
    df['region'] = df['label'].str.split('_').str[-1]
    
    # Map the 'final.rank' from SA_rank to the dataframe
    df['final.rank'] = df['region'].map(region_to_rank)
    
    # Drop the temporary 'region' column if not needed
    df.drop(columns=['region'], inplace=True)

## Inspect the second-derivative distribution of both tables

In [ ]:
# Analyze the face data
# Cumulative distribution plot
plt.subplot(2, 2, 4)
sorted_values = np.sort(face['Second_Mean'])
plt.plot(sorted_values, np.arange(1, len(sorted_values) + 1) / len(sorted_values))
plt.title('Cumulative Distribution')
plt.xlabel('Second_Mean')
plt.ylabel('Cumulative Probability')

plt.tight_layout()
plt.show()

# Count points retained under different thresholds
thresholds = [0.001, 0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008, 0.009, 0.01, 0.02, 0.03, 0.04, 0.05]

print("Threshold analysis for filtering points near 0:")
print("Threshold\tCount < threshold\tPercentage\tCount >= threshold")
print("-" * 60)

for threshold in thresholds:
    count_below = (np.abs(face['Second_Mean']) < threshold).sum()
    percentage = (count_below / len(face)) * 100
    count_above = len(face) - count_below
    print(f"{threshold:.3f}\t\t{count_below}\t\t\t{percentage:.1f}%\t\t{count_above}")

# Show basic summary statistics
print(f"\nBasic statistics:")
print(f"Mean: {face['Second_Mean'].mean():.6f}")
print(f"Median: {face['Second_Mean'].median():.6f}")
print(f"Std: {face['Second_Mean'].std():.6f}")
print(f"Min: {face['Second_Mean'].min():.6f}")
print(f"Max: {face['Second_Mean'].max():.6f}")

In [ ]:
# Analyze the place data
# Cumulative distribution plot
plt.subplot(2, 2, 4)
sorted_values = np.sort(place['Second_Mean'])
plt.plot(sorted_values, np.arange(1, len(sorted_values) + 1) / len(sorted_values))
plt.title('Cumulative Distribution')
plt.xlabel('Second_Mean')
plt.ylabel('Cumulative Probability')

plt.tight_layout()
plt.show()

# Count points retained under different thresholds
thresholds = [0.001, 0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008, 0.009, 0.01, 0.02, 0.03, 0.04, 0.05]

print("Threshold analysis for filtering points near 0:")
print("Threshold\tCount < threshold\tPercentage\tCount >= threshold")
print("-" * 60)

for threshold in thresholds:
    count_below = (np.abs(place['Second_Mean']) < threshold).sum()
    percentage = (count_below / len(place)) * 100
    count_above = len(place) - count_below
    print(f"{threshold:.3f}\t\t{count_below}\t\t\t{percentage:.1f}%\t\t{count_above}")

# Show basic summary statistics
print(f"\nBasic statistics:")
print(f"Mean: {place['Second_Mean'].mean():.6f}")
print(f"Median: {place['Second_Mean'].median():.6f}")
print(f"Std: {place['Second_Mean'].std():.6f}")
print(f"Min: {place['Second_Mean'].min():.6f}")
print(f"Max: {place['Second_Mean'].max():.6f}")

## Gaussian Mixture Model point selection

In [ ]:
from sklearn.mixture import GaussianMixture
from scipy.stats import spearmanr

In [ ]:
def gaussian_mixture_threshold(data, n_components=2):
    """
    Use a Gaussian Mixture Model to automatically identify points near zero
    """
    # Reshape the data into a column vector
    X = data.values.reshape(-1, 1)
    
    # Fit the Gaussian Mixture Model
    gmm = GaussianMixture(n_components=n_components, random_state=42)
    gmm.fit(X)
    
    # Predict which component each point belongs to
    labels = gmm.predict(X)
    
    # Get the mean and variance of each component
    means = gmm.means_.flatten()
    covariances = gmm.covariances_.flatten()
    weights = gmm.weights_
    
    # Identify which component is closer to zero
    zero_component = np.argmin(np.abs(means))
    
    print(f"Component {zero_component} (near zero): mean={means[zero_component]:.6f}, std={np.sqrt(covariances[zero_component]):.6f}, weight={weights[zero_component]:.3f}")
    print(f"Component {1-zero_component} (significant): mean={means[1-zero_component]:.6f}, std={np.sqrt(covariances[1-zero_component]):.6f}, weight={weights[1-zero_component]:.3f}")
    
    # Return the mask for the non-zero (non-linear) component
    significant_mask = labels != zero_component
    
    return significant_mask, gmm

# Apply to the face data
significant_mask_face, gmm_face = gaussian_mixture_threshold(face['Second_Mean'])
print(f"\nFace data: {significant_mask_face.sum()} non-linear points / {len(face)} total ({significant_mask_face.sum()/len(face)*100:.1f}%)")

# Apply to the place data
significant_mask_place, gmm_place = gaussian_mixture_threshold(place['Second_Mean'])
print(f"Place data: {significant_mask_place.sum()} non-linear points / {len(place)} total ({significant_mask_place.sum()/len(place)*100:.1f}%)")

In [ ]:
def plot_mixture_results(data, mask, gmm, title):
    """
    Visualize the mixture-model result
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # Raw distribution + fitted mixture model
    ax1.hist(data, bins=50, density=True, alpha=0.7, color='lightblue')
    
    # Plot the mixture-model components
    x_range = np.linspace(data.min(), data.max(), 1000)
    for i in range(gmm.n_components):
        mean = gmm.means_[i, 0]
        std = np.sqrt(gmm.covariances_[i, 0])
        weight = gmm.weights_[i]
        
        # Individual Gaussian component
        gaussian = weight * (1/np.sqrt(2*np.pi*std**2)) * np.exp(-0.5*((x_range-mean)/std)**2)
        ax1.plot(x_range, gaussian, '--', label=f'Component {i}')
    
    # Overall mixture distribution
    log_prob = gmm.score_samples(x_range.reshape(-1, 1))
    ax1.plot(x_range, np.exp(log_prob), 'k-', linewidth=2, label='Mixture')
    
    ax1.set_title(f'{title} - Mixture Model Fit')
    ax1.legend()
    
    # Comparison of the filtered result
    ax2.hist(data[~mask], bins=30, alpha=0.7, label='Near Zero', color='red')
    ax2.hist(data[mask], bins=30, alpha=0.7, label='Significant', color='green')
    ax2.set_title(f'{title} - Filtered Results')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

# Visualize the result
plot_mixture_results(face['Second_Mean'], significant_mask_face, gmm_face, 'Face Data')
plot_mixture_results(place['Second_Mean'], significant_mask_place, gmm_place, 'Place Data')

In [ ]:
significant_face = face[significant_mask_face]  
significant_place = place[significant_mask_place]

In [ ]:
# Counts
# significant_face[significant_face['First_Mean'] < 0].count()
significant_place[significant_place['First_Mean'] < 0].count()

In [ ]:
significant_face.to_csv('data/GMM_Face_full_results.csv', index=False)
significant_place.to_csv('data/GMM_Place_full_results.csv', index=False)

In [ ]:
significant_face

### Regression test: original

In [ ]:
# Run regression on the filtered result: place
# Fit the model for significant_place
X = sm.add_constant(significant_place['final.rank'])  # Add a constant for the intercept
model = sm.OLS(significant_place['Second_Mean'], X).fit()

# Plot the fitted line
plt.scatter(significant_place['final.rank'], significant_place['Second_Mean'])
plt.plot(significant_place['final.rank'], model.predict(X), label='Fitted Line', color='red')

plt.xlabel('Final Rank')
plt.ylabel('Second Mean')
plt.title('Linear Regression: Second Mean vs Final Rank (place filtered)')
plt.legend()
plt.show()

# Output the model summary
print(model.summary())

# Calculate Spearman's rho
rho, p_value = spearmanr(significant_place['Second_Mean'], significant_place['final.rank'])
print(f"Spearman's rho: {rho}, p-value: {p_value}")

In [ ]:
# Run regression on the filtered result: place
# Fit the model for significant_place
X = sm.add_constant(significant_place['final.rank'])  # Add a constant for the intercept
model = sm.OLS(significant_place['Second_Mean'], X).fit()

# Plot the fitted line
plt.scatter(significant_place['final.rank'], significant_place['Second_Mean'])
plt.plot(significant_place['final.rank'], model.predict(X), label='Fitted Line', color='red')

plt.xlabel('Final Rank')
plt.ylabel('Second Mean')
plt.title('Linear Regression: Second Mean vs Final Rank (place filtered)')
plt.legend()
plt.show()

# Output the model summary
print(model.summary())

# Calculate Spearman's rho
rho, p_value = spearmanr(significant_place['Second_Mean'], significant_place['final.rank'])
print(f"Spearman's rho: {rho}, p-value: {p_value}")

### Regression test: split by sign of the first derivative

In [ ]:
# Split the significant_face data by the sign of First_Mean
positive_first_face = significant_face[significant_face['First_Mean'] > 0]
negative_first_face = significant_face[significant_face['First_Mean'] < 0]

print(f"Positive First_Mean: {len(positive_first_face)} regions")
print(f"Negative First_Mean: {len(negative_first_face)} regions")

# Regression on the First_Mean > 0 subset
if len(positive_first_face) > 0:
    print("\n=== Analysis for regions with positive First_Mean ===")
    X_pos = sm.add_constant(positive_first_face['final.rank'])
    model_pos = sm.OLS(positive_first_face['Second_Mean'], X_pos).fit()
    
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.scatter(positive_first_face['final.rank'], positive_first_face['Second_Mean'])
    plt.plot(positive_first_face['final.rank'], model_pos.predict(X_pos), label='Fitted Line', color='red')
    plt.xlabel('Final Rank')
    plt.ylabel('Second Mean')
    plt.title('Linear Regression: Second Mean vs Final Rank (Face, First_Mean > 0)')
    plt.legend()
    
    print(model_pos.summary())
    
    rho_pos, p_value_pos = spearmanr(positive_first_face['Second_Mean'], positive_first_face['final.rank'])
    print(f"Spearman's rho: {rho_pos}, p-value: {p_value_pos}")

# Regression on the First_Mean < 0 subset
if len(negative_first_face) > 0:
    print("\n=== Analysis for regions with negative First_Mean ===")
    X_neg = sm.add_constant(negative_first_face['final.rank'])
    model_neg = sm.OLS(negative_first_face['Second_Mean'], X_neg).fit()
    
    if len(positive_first_face) > 0:
        plt.subplot(1, 2, 2)
    else:
        plt.figure(figsize=(6, 5))
    
    plt.scatter(negative_first_face['final.rank'], negative_first_face['Second_Mean'])
    plt.plot(negative_first_face['final.rank'], model_neg.predict(X_neg), label='Fitted Line', color='red')
    plt.xlabel('Final Rank')
    plt.ylabel('Second Mean')
    plt.title('Linear Regression: Second Mean vs Final Rank (Face, First_Mean < 0)')
    plt.legend()
    
    plt.tight_layout()
    plt.show()
    
    print(model_neg.summary())
    
    rho_neg, p_value_neg = spearmanr(negative_first_face['Second_Mean'], negative_first_face['final.rank'])
    print(f"Spearman's rho: {rho_neg}, p-value: {p_value_neg}")

if len(positive_first_face) > 0 and len(negative_first_face) == 0:
    plt.show()

In [ ]:
# Split the significant_place data by the sign of First_Mean
positive_first_place = significant_place[significant_place['First_Mean'] > 0]
negative_first_place = significant_place[significant_place['First_Mean'] < 0]

print(f"Positive First_Mean: {len(positive_first_place)} regions")
print(f"Negative First_Mean: {len(negative_first_place)} regions")

# Regression on the First_Mean > 0 subset
if len(positive_first_place) > 0:
    print("\n=== Analysis for regions with positive First_Mean ===")
    X_pos = sm.add_constant(positive_first_place['final.rank'])
    model_pos = sm.OLS(positive_first_place['Second_Mean'], X_pos).fit()
    
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.scatter(positive_first_place['final.rank'], positive_first_place['Second_Mean'])
    plt.plot(positive_first_place['final.rank'], model_pos.predict(X_pos), label='Fitted Line', color='red')
    plt.xlabel('Final Rank')
    plt.ylabel('Second Mean')
    plt.title('Linear Regression: Second Mean vs Final Rank (place, First_Mean > 0)')
    plt.legend()
    
    print(model_pos.summary())
    
    rho_pos, p_value_pos = spearmanr(positive_first_place['Second_Mean'], positive_first_place['final.rank'])
    print(f"Spearman's rho: {rho_pos}, p-value: {p_value_pos}")

# Regression on the First_Mean < 0 subset
if len(negative_first_place) > 0:
    print("\n=== Analysis for regions with negative First_Mean ===")
    X_neg = sm.add_constant(negative_first_place['final.rank'])
    model_neg = sm.OLS(negative_first_place['Second_Mean'], X_neg).fit()
    
    if len(positive_first_place) > 0:
        plt.subplot(1, 2, 2)
    else:
        plt.figure(figsize=(6, 5))
    
    plt.scatter(negative_first_place['final.rank'], negative_first_place['Second_Mean'])
    plt.plot(negative_first_place['final.rank'], model_neg.predict(X_neg), label='Fitted Line', color='red')
    plt.xlabel('Final Rank')
    plt.ylabel('Second Mean')
    plt.title('Linear Regression: Second Mean vs Final Rank (place, First_Mean < 0)')
    plt.legend()
    
    plt.tight_layout()
    plt.show()
    
    print(model_neg.summary())
    
    rho_neg, p_value_neg = spearmanr(negative_first_place['Second_Mean'], negative_first_place['final.rank'])
    print(f"Spearman's rho: {rho_neg}, p-value: {p_value_neg}")

if len(positive_first_place) > 0 and len(negative_first_place) == 0:
    plt.show()

### Regression test: two randomly split subsamples

In [ ]:
face_sample1 = pd.read_csv('data/face_subsample1_derivatives_results_label_with_rank.csv')
face_sample2 = pd.read_csv('data/face_subsample2_derivatives_results_label_with_rank.csv')
place_sample1 = pd.read_csv('data/place_subsample1_derivatives_results_label_with_rank.csv')
place_sample2 = pd.read_csv('data/place_subsample2_derivatives_results_label_with_rank.csv')

In [ ]:
body_sample1 = pd.read_csv('data/body_subsample1_derivatives_results_label_with_rank.csv')
body_sample2 = pd.read_csv('data/body_subsample2_derivatives_results_label_with_rank.csv')
tool_sample1 = pd.read_csv('data/tool_subsample1_derivatives_results_label_with_rank.csv')
tool_sample2 = pd.read_csv('data/tool_subsample2_derivatives_results_label_with_rank.csv')

In [ ]:
# Apply GMM filtering and regression to the four newly loaded subsamples

datasets = {
    'face_sample1': face_sample1,
    'face_sample2': face_sample2,
    'place_sample1': place_sample1,
    'place_sample2': place_sample2
}

for dataset_name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"Analysis for {dataset_name}")
    print(f"{'='*60}")
    
    # Apply GMM filtering
    significant_mask, gmm = gaussian_mixture_threshold(df['Second_Mean'])
    significant_data = df[significant_mask]
    # significant_data.to_csv(f'GMM_{dataset_name}_full_results.csv', index=False)
    
    print(f"Filtered: {significant_mask.sum()} non-linear points / {len(df)} total ({significant_mask.sum()/len(df)*100:.1f}%)")
    
    # Regression on the filtered data
    print(f"\n--- Overall regression ({dataset_name}) ---")
    X = sm.add_constant(significant_data['final.rank'])
    model = sm.OLS(significant_data['Second_Mean'], X).fit()
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.scatter(significant_data['final.rank'], significant_data['Second_Mean'], alpha=0.6)
    plt.plot(significant_data['final.rank'], model.predict(X), label='Fitted Line', color='red')
    plt.xlabel('Final Rank')
    plt.ylabel('Second Mean')
    plt.title(f'Linear Regression: Second Mean vs Final Rank ({dataset_name})')
    plt.legend()
    plt.show()
    
    # Print the statistics
    print(f"R-squared: {model.rsquared:.4f}")
    print(f"P-value (slope): {model.pvalues[1]:.6f}")
    
    # Spearman correlation
    rho, p_value = spearmanr(significant_data['Second_Mean'], significant_data['final.rank'])
    print(f"Spearman's rho: {rho:.4f}, p-value: {p_value:.6f}")
    
    # Group analysis by sign of the first derivative
    positive_first = significant_data[significant_data['First_Mean'] > 0]
    negative_first = significant_data[significant_data['First_Mean'] < 0]
    
    print(f"\n--- Grouped analysis ({dataset_name}) ---")
    print(f"Positive First_Mean: {len(positive_first)} regions")
    print(f"Negative First_Mean: {len(negative_first)} regions")
    
    if len(positive_first) > 5:  # need enough data points
        X_pos = sm.add_constant(positive_first['final.rank'])
        model_pos = sm.OLS(positive_first['Second_Mean'], X_pos).fit()
        rho_pos, p_pos = spearmanr(positive_first['Second_Mean'], positive_first['final.rank'])
        print(f"Positive group - R²: {model_pos.rsquared:.4f}, p-value: {model_pos.pvalues[1]:.6f}, Spearman's rho: {rho_pos:.4f}")
    
    if len(negative_first) > 5:
        X_neg = sm.add_constant(negative_first['final.rank'])
        model_neg = sm.OLS(negative_first['Second_Mean'], X_neg).fit()
        rho_neg, p_neg = spearmanr(negative_first['Second_Mean'], negative_first['final.rank'])
        print(f"Negative group - R²: {model_neg.rsquared:.4f}, p-value: {model_neg.pvalues[1]:.6f}, Spearman's rho: {rho_neg:.4f}")

print(f"\n{'='*60}")
print("Analysis complete")
print(f"{'='*60}")

In [ ]:
# Apply GMM filtering and regression to the four subsamples, body & tool

datasets = {
    'body_sample1': body_sample1,
    'body_sample2': body_sample2,
    'tool_sample1': tool_sample1,
    'tool_sample2': tool_sample2
}

for dataset_name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"Analysis for {dataset_name}")
    print(f"{'='*60}")
    
    # Apply GMM filtering
    significant_mask, gmm = gaussian_mixture_threshold(df['Second_Mean'])
    significant_data = df[significant_mask]
    # significant_data.to_csv(f'GMM_{dataset_name}_full_results.csv', index=False)
    
    print(f"Filtered: {significant_mask.sum()} non-linear points / {len(df)} total ({significant_mask.sum()/len(df)*100:.1f}%)")
    
    # Regression on the filtered data
    print(f"\n--- Overall regression ({dataset_name}) ---")
    X = sm.add_constant(significant_data['final.rank'])
    model = sm.OLS(significant_data['Second_Mean'], X).fit()
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.scatter(significant_data['final.rank'], significant_data['Second_Mean'], alpha=0.6)
    plt.plot(significant_data['final.rank'], model.predict(X), label='Fitted Line', color='red')
    plt.xlabel('Final Rank')
    plt.ylabel('Second Mean')
    plt.title(f'Linear Regression: Second Mean vs Final Rank ({dataset_name})')
    plt.legend()
    plt.show()
    
    # Print the statistics
    print(f"R-squared: {model.rsquared:.4f}")
    print(f"P-value (slope): {model.pvalues[1]:.6f}")
    
    # Spearman correlation
    rho, p_value = spearmanr(significant_data['Second_Mean'], significant_data['final.rank'])
    print(f"Spearman's rho: {rho:.4f}, p-value: {p_value:.6f}")
    
    # Group analysis by sign of the first derivative
    positive_first = significant_data[significant_data['First_Mean'] > 0]
    negative_first = significant_data[significant_data['First_Mean'] < 0]
    
    print(f"\n--- Grouped analysis ({dataset_name}) ---")
    print(f"Positive First_Mean: {len(positive_first)} regions")
    print(f"Negative First_Mean: {len(negative_first)} regions")
    
    if len(positive_first) > 5:  # need enough data points
        X_pos = sm.add_constant(positive_first['final.rank'])
        model_pos = sm.OLS(positive_first['Second_Mean'], X_pos).fit()
        rho_pos, p_pos = spearmanr(positive_first['Second_Mean'], positive_first['final.rank'])
        print(f"Positive group - R²: {model_pos.rsquared:.4f}, p-value: {model_pos.pvalues[1]:.6f}, Spearman's rho: {rho_pos:.4f}")
    
    if len(negative_first) > 5:
        X_neg = sm.add_constant(negative_first['final.rank'])
        model_neg = sm.OLS(negative_first['Second_Mean'], X_neg).fit()
        rho_neg, p_neg = spearmanr(negative_first['Second_Mean'], negative_first['final.rank'])
        print(f"Negative group - R²: {model_neg.rsquared:.4f}, p-value: {model_neg.pvalues[1]:.6f}, Spearman's rho: {rho_neg:.4f}")

print(f"\n{'='*60}")
print("Analysis complete")
print(f"{'='*60}")

### Regression test: GAM-significant regions

In [ ]:
faceSigGAM = pd.read_csv('data/face_results_with_rank.csv')
placeSigGAM = pd.read_csv('data/place_results_with_rank.csv')

In [ ]:
# Apply GMM filtering and regression to the four newly loaded subsamples

datasets = {
    'face_FDR_GAM': faceSigGAM,
    'place_FDR_GAM': placeSigGAM,
}

for dataset_name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"Analysis for {dataset_name}")
    print(f"{'='*60}")
    
    # Apply GMM filtering
    significant_mask, gmm = gaussian_mixture_threshold(df['Second_Mean'])
    significant_data = df[significant_mask]
    
    print(f"Filtered: {significant_mask.sum()} non-linear points / {len(df)} total ({significant_mask.sum()/len(df)*100:.1f}%)")
    
    # Regression on the filtered data
    print(f"\n--- Overall regression ({dataset_name}) ---")
    X = sm.add_constant(significant_data['final.rank'])
    model = sm.OLS(significant_data['Second_Mean'], X).fit()
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.scatter(significant_data['final.rank'], significant_data['Second_Mean'], alpha=0.6)
    plt.plot(significant_data['final.rank'], model.predict(X), label='Fitted Line', color='red')
    plt.xlabel('Final Rank')
    plt.ylabel('Second Mean')
    plt.title(f'Linear Regression: Second Mean vs Final Rank ({dataset_name})')
    plt.legend()
    plt.show()
    
    # Print the statistics
    print(f"R-squared: {model.rsquared:.4f}")
    print(f"P-value (slope): {model.pvalues[1]:.6f}")
    
    # Spearman correlation
    rho, p_value = spearmanr(significant_data['Second_Mean'], significant_data['final.rank'])
    print(f"Spearman's rho: {rho:.4f}, p-value: {p_value:.6f}")
    
    # Group analysis by sign of the first derivative
    positive_first = significant_data[significant_data['First_Mean'] > 0]
    negative_first = significant_data[significant_data['First_Mean'] < 0]
    
    print(f"\n--- Grouped analysis ({dataset_name}) ---")
    print(f"Positive First_Mean: {len(positive_first)} regions")
    print(f"Negative First_Mean: {len(negative_first)} regions")
    
    if len(positive_first) > 5:  # need enough data points
        X_pos = sm.add_constant(positive_first['final.rank'])
        model_pos = sm.OLS(positive_first['Second_Mean'], X_pos).fit()
        rho_pos, p_pos = spearmanr(positive_first['Second_Mean'], positive_first['final.rank'])
        print(f"Positive group - R²: {model_pos.rsquared:.4f}, p-value: {model_pos.pvalues[1]:.6f}, Spearman's rho: {rho_pos:.4f}")
    
    if len(negative_first) > 5:
        X_neg = sm.add_constant(negative_first['final.rank'])
        model_neg = sm.OLS(negative_first['Second_Mean'], X_neg).fit()
        rho_neg, p_neg = spearmanr(negative_first['Second_Mean'], negative_first['final.rank'])
        print(f"Negative group - R²: {model_neg.rsquared:.4f}, p-value: {model_neg.pvalues[1]:.6f}, Spearman's rho: {rho_neg:.4f}")

print(f"\n{'='*60}")
print("Analysis complete")
print(f"{'='*60}")

In [ ]:
# Apply GMM filtering to the GAM-significant results and save
print("Applying GMM filtering to the GAM-significant results...")

# Apply GMM filtering to faceSigGAM
print("\n=== Face GAM-significant results: GMM filtering ===")
significant_mask_faceSigGAM, gmm_faceSigGAM = gaussian_mixture_threshold(faceSigGAM['Second_Mean'])
GMM_FaceSig = faceSigGAM[significant_mask_faceSigGAM]
print(f"Face GAM filtered: {significant_mask_faceSigGAM.sum()} non-linear points / {len(faceSigGAM)} total ({significant_mask_faceSigGAM.sum()/len(faceSigGAM)*100:.1f}%)")

# Apply GMM filtering to placeSigGAM
print("\n=== Place GAM-significant results: GMM filtering ===")
significant_mask_placeSigGAM, gmm_placeSigGAM = gaussian_mixture_threshold(placeSigGAM['Second_Mean'])
GMM_PlaceSig = placeSigGAM[significant_mask_placeSigGAM]
print(f"Place GAM filtered: {significant_mask_placeSigGAM.sum()} non-linear points / {len(placeSigGAM)} total ({significant_mask_placeSigGAM.sum()/len(placeSigGAM)*100:.1f}%)")

# Save the filtered results
GMM_FaceSig.to_csv('data/GMM_FaceSig.csv', index=False)
GMM_PlaceSig.to_csv('data/GMM_PlaceSig.csv', index=False)

print(f"\nFiltered results saved:")
print(f"- GMM_FaceSig.csv: {len(GMM_FaceSig)} regions")
print(f"- GMM_PlaceSig.csv: {len(GMM_PlaceSig)} regions")

# Show basic info on the filtered data
print(f"\n=== GMM_FaceSig summary ===")
print(f"shape: {GMM_FaceSig.shape}")
print(f"Second_Mean range: {GMM_FaceSig['Second_Mean'].min():.6f} to {GMM_FaceSig['Second_Mean'].max():.6f}")

print(f"\n=== GMM_PlaceSig summary ===")
print(f"shape: {GMM_PlaceSig.shape}")
print(f"Second_Mean range: {GMM_PlaceSig['Second_Mean'].min():.6f} to {GMM_PlaceSig['Second_Mean'].max():.6f}")

## Wilcoxon Rank-Sum Test Permutation Distribution

**Task:**
- Obtain the permutation distribution of the Wilcoxon rank-sum statistic under the null (no difference in location)
- Based on two independent samples of size m=4 and n=3
- Plot a histogram of the permutation distribution
- Assess whether the distribution is symmetric

**Background:**
- With total sample size N=m+n=7, the ranks of all observations are 1,2,3,4,5,6,7
- The Wilcoxon rank-sum statistic W is the sum of ranks of the first group (m=4)
- Under the null, all possible rank assignments are equally likely
- There are C(7,4) = 35 possible assignments in total

In [ ]:
# Wilcoxon rank-sum test permutation distribution
# Sample sizes: m=4, n=3

# Load required libraries
if (!require(combinat)) install.packages("combinat")
if (!require(moments)) install.packages("moments")
library(combinat)
library(moments)

# Function to calculate Wilcoxon rank-sum statistic
wilcoxon_rank_sum <- function(group1_ranks, total_ranks = 1:7) {
  # The Wilcoxon rank-sum statistic is the sum of ranks in the first group
  sum(group1_ranks)
}

# Generate all possible combinations of 4 ranks from 7 total ranks
# (This represents all possible ways to assign ranks to group 1)
all_combinations <- combn(7, 4, simplify = FALSE)

# Calculate the rank-sum statistic for each permutation
W_values <- sapply(all_combinations, wilcoxon_rank_sum)

# Display the permutation distribution
cat("Permutation distribution of Wilcoxon rank-sum statistic:\n")
cat("Sample sizes: m =", 4, ", n =", 3, "\n")
cat("Total number of permutations:", length(W_values), "\n\n")

# Show frequency table
freq_table <- table(W_values)
cat("Frequency distribution:\n")
print(freq_table)

# Create histogram
par(mfrow = c(1, 2))

# Histogram
hist(W_values, 
     breaks = seq(min(W_values) - 0.5, max(W_values) + 0.5, by = 1),
     main = "Permutation Distribution of Wilcoxon Rank-Sum Statistic",
     xlab = "Wilcoxon Rank-Sum Statistic (W)",
     ylab = "Frequency",
     col = "lightblue",
     border = "black",
     xlim = c(9, 23))

# Add vertical line at the mean
abline(v = mean(W_values), col = "red", lwd = 2, lty = 2)
text(mean(W_values), max(freq_table) * 0.8, 
     paste("Mean =", round(mean(W_values), 2)), 
     col = "red", pos = 4)

# Box plot to better show symmetry
boxplot(W_values, 
        main = "Box Plot of Permutation Distribution",
        ylab = "Wilcoxon Rank-Sum Statistic (W)",
        col = "lightgreen")

par(mfrow = c(1, 1))

# Check for symmetry
cat("\nSymmetry Analysis:\n")
cat("Mean:", round(mean(W_values), 3), "\n")
cat("Median:", median(W_values), "\n")
cat("Standard Deviation:", round(sd(W_values), 3), "\n")
cat("Skewness:", round(skewness(W_values), 3), "\n")

# Test for symmetry around the mean
theoretical_mean <- 4 * (4 + 3 + 1) / 2  # m * (m + n + 1) / 2
cat("Theoretical mean:", theoretical_mean, "\n")

# Check if distribution is symmetric
is_symmetric <- abs(mean(W_values) - median(W_values)) < 0.001
cat("Is the distribution symmetric? ", ifelse(is_symmetric, "Yes", "No"), "\n")

# Display all possible W values and their frequencies
cat("\nDetailed frequency table:\n")
for (w in sort(unique(W_values))) {
  cat("W =", w, ": frequency =", sum(W_values == w), "\n")
}